## Exercise 1 : Manual TF-IDF pipeline


In [1]:
import re
import numpy as np
import pandas as pd

docs_raw = [
    "The Cat chased the Mouse.",
    "The dog Barked at the Cat.",
    "The mouse ran away from the Cat.",
]

def tokenize(s):
    s = s.lower()
    s = re.sub(r"[^a-z\s]", " ", s)
    return [t for t in s.split() if t]

tokens_per_doc = [tokenize(d) for d in docs_raw]
print("Tokens (stop words kept, Exercise 1):")
for i, toks in enumerate(tokens_per_doc, 1):
    print(f"  D{i}: {toks}")

vocab = sorted({t for doc in tokens_per_doc for t in doc})
word2i = {w: i for i, w in enumerate(vocab)}
N_docs = len(tokens_per_doc)
V = len(vocab)

counts = np.zeros((N_docs, V))
for d, toks in enumerate(tokens_per_doc):
    for t in toks:
        counts[d, word2i[t]] += 1

TF = counts / counts.sum(axis=1, keepdims=True)

def wtf_matrix(tf):
    out = np.zeros_like(tf)
    m = tf > 0
    out[m] = 1 + np.log10(tf[m])
    return out

WTF = wtf_matrix(TF)
df = (counts > 0).sum(axis=0)
IDF = np.log10(N_docs / (1 + df))
TFIDF = TF * IDF

idx = [f"D{i+1}" for i in range(N_docs)]
print("\nVocabulary (sorted):", vocab)
print("\nTF:")
print(pd.DataFrame(TF, index=idx, columns=vocab).round(6))
print("\nNormalized TF (1 + log10 TF):")
print(pd.DataFrame(WTF, index=idx, columns=vocab).round(4))
print("\nIDF:")
print(pd.Series(IDF, index=vocab).round(4))
print("\nTF-IDF (TF × IDF):")
print(pd.DataFrame(TFIDF, index=idx, columns=vocab).round(4))

Tokens (stop words kept, Exercise 1):
  D1: ['the', 'cat', 'chased', 'the', 'mouse']
  D2: ['the', 'dog', 'barked', 'at', 'the', 'cat']
  D3: ['the', 'mouse', 'ran', 'away', 'from', 'the', 'cat']

Vocabulary (sorted): ['at', 'away', 'barked', 'cat', 'chased', 'dog', 'from', 'mouse', 'ran', 'the']

TF:
          at      away    barked       cat  chased       dog      from  \
D1  0.000000  0.000000  0.000000  0.200000     0.2  0.000000  0.000000   
D2  0.166667  0.000000  0.166667  0.166667     0.0  0.166667  0.000000   
D3  0.000000  0.142857  0.000000  0.142857     0.0  0.000000  0.142857   

       mouse       ran       the  
D1  0.200000  0.000000  0.400000  
D2  0.000000  0.000000  0.333333  
D3  0.142857  0.142857  0.285714  

Normalized TF (1 + log10 TF):
        at    away  barked     cat  chased     dog    from   mouse     ran  \
D1  0.0000  0.0000  0.0000  0.3010   0.301  0.0000  0.0000  0.3010  0.0000   
D2  0.2218  0.0000  0.2218  0.2218   0.000  0.2218  0.0000  0.0000  0.000

**Exercise 1 : Discussion**

1. **TF-IDF vs raw TF** : Raw TF just counts how many times a word appears, so common words (like the, and) can look important even when they are not. TF-IDF fixes this by reducing the importance of words that appear in many documents and giving more importance to words that are more unique. So it helps highlight what makes each document different.

2. **Higher importance** : A word gets a high TF-IDF score when it appears a lot in one document but not in many others. If a word shows up in almost every document, its importance goes down, and sometimes it can even get a negative value depending on the formula used.

3. **More documents** : When we add more documents, the importance of words can change. Some words may become more common or more rare, so their scores and rankings will shift.

In [2]:
# Exercise 2
vocab_cbow = ["barked", "cat", "chased", "dog", "mouse", "ran"]
wi = {w: i for i, w in enumerate(vocab_cbow)}

def cbow_pairs(seq, window=1):
    rows = []
    for i, target in enumerate(seq):
        ctx = []
        if i - window >= 0:
            ctx.append(seq[i - window])
        if i + window < len(seq):
            ctx.append(seq[i + window])
        rows.append({
            "context_words": ctx,
            "context_indices": [wi[w] for w in ctx],
            "target": target,
            "target_index": wi[target],
        })
    return pd.DataFrame(rows)

D1 = ["cat", "chased", "mouse"]
D2 = ["dog", "barked", "cat"]
D3 = ["mouse", "ran", "cat"]

print("D1\n", cbow_pairs(D1), "\n")
print("D2\n", cbow_pairs(D2), "\n")
print("D3\n", cbow_pairs(D3), "\n")

oh = np.eye(len(vocab_cbow))
one_hot_df = pd.DataFrame(oh, index=vocab_cbow, columns=[f"h{i}" for i in range(len(vocab_cbow))])
print("One-hot rows:\n", one_hot_df.astype(int))

D1
   context_words context_indices  target  target_index
0      [chased]             [2]     cat             1
1  [cat, mouse]          [1, 4]  chased             2
2      [chased]             [2]   mouse             4 

D2
   context_words context_indices  target  target_index
0      [barked]             [0]     dog             3
1    [dog, cat]          [3, 1]  barked             0
2      [barked]             [0]     cat             1 

D3
   context_words context_indices target  target_index
0         [ran]             [5]  mouse             4
1  [mouse, cat]          [4, 1]    ran             5
2         [ran]             [5]    cat             1 

One-hot rows:
         h0  h1  h2  h3  h4  h5
barked   1   0   0   0   0   0
cat      0   1   0   0   0   0
chased   0   0   1   0   0   0
dog      0   0   0   1   0   0
mouse    0   0   0   0   1   0
ran      0   0   0   0   0   1


**Exercise 2 : Discussion**

1. **Same word, different contexts (cat)** : Every time the word appears in a different sentence, it learns from that new context. So “cat” doesn’t just learn from one situation, it learns from many different surrounding words. Over time, it builds a general understanding instead of depending on just one set of nearby words.

2. **Distributional hypothesis** : The idea is simple: words that appear in similar situations usually have similar meanings. In CBOW, the model tries to guess a word based on the words around it. Because of this, words that appear in similar contexts end up having similar meanings in the model.

**CBOW architecture (summary)**

- **Input layer:** Each word is turned into a one-hot vector of size 6 (since there are 6 words total). If you have two context words, you use two one-hot vectors.
- **Hidden layer:** This layer creates the word embedding (size 4). It takes the vectors for the context words from the input weights and averages them. There’s no activation function here, it’s just averaging.
- **Output layer:** The averaged vector is multiplied by another weight matrix to get 6 output values (one for each word). Then softmax is used to turn these into probabilities.
- **After training:** The input weight matrix is what we keep, because its rows become the word embeddings. The output weights are usually not needed anymore.


In [3]:
# Tutorial 8
def softmax(x):
    x = np.asarray(x, dtype=float)
    e = np.exp(x - np.max(x))
    return e / e.sum()

W_input = np.array([
    [0.1, 0.2, -0.1, 0.0],
    [0.4, 0.5, -0.1, 0.2],
    [-0.2, 0.3, 0.2, -0.1],
    [0.3, -0.1, 0.1, 0.0],
    [0.5, -0.4, 0.1, 0.1],
    [-0.3, 0.0, -0.2, 0.3],
])
W_hidden = np.array([
    [0.2, -0.1, 0.3, 0.1, 0.4, 0.0],
    [0.1, 0.3, -0.2, -0.1, 0.2, 0.5],
    [-0.4, 0.5, 0.1, -0.2, -0.1, 0.3],
    [0.1, -0.2, -0.4, 0.3, -0.3, 0.1],
])

idx_cat, idx_mouse, idx_chased = 1, 4, 2
v_cat = W_input[idx_cat]
v_mouse = W_input[idx_mouse]
v_hat = (v_cat + v_mouse) / 2.0
# (N,) @ (N×V) → (V,) logits — matches handout dimensions
logits = v_hat @ W_hidden
probs = softmax(logits)
vocab6 = ["barked", "cat", "chased", "dog", "mouse", "ran"]

print("Average hidden vector v̂:", np.round(v_hat, 4))
print("Logits z:", np.round(logits, 4))
print("Softmax P(word|context):", np.round(probs, 4))
pred_i = int(np.argmax(logits))
print("Argmax (random init):", vocab6[pred_i], "| true target: chased — SGD + CE loss aligns prediction over training.")

Average hidden vector v̂: [0.45 0.05 0.   0.15]
Logits z: [ 0.11  -0.06   0.065  0.085  0.145  0.04 ]
Softmax P(word|context): [0.1741 0.1469 0.1665 0.1698 0.1803 0.1624]
Argmax (random init): mouse | true target: chased — SGD + CE loss aligns prediction over training.
